In [2]:
import torch
import pandas as pd
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

from sklearn.metrics import accuracy_score, f1_score

print("All imports successful!")
print("PyTorch:", torch.__version__)

All imports successful!
PyTorch: 2.13.0+cpu


In [3]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")
test_df = pd.read_csv("../data/processed/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nColumns:")
print(train_df.columns)

Train: (1728, 3)
Validation: (370, 3)
Test: (371, 3)

Columns:
Index(['file_content', 'document_type', 'label'], dtype='str')


In [4]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded!")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded!
Vocabulary size: 30522


In [5]:
def tokenize_data(df):
    return tokenizer(
        df["file_content"].tolist(),
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_encodings = tokenize_data(train_df)
val_encodings = tokenize_data(val_df)
test_encodings = tokenize_data(test_df)

print("Tokenization complete!")
print("Training examples:", len(train_encodings["input_ids"]))
print("Tokens per example:", len(train_encodings["input_ids"][0]))

Tokenization complete!
Training examples: 1728
Tokens per example: 256


In [6]:
class DocumentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [7]:
train_dataset = DocumentDataset(
    train_encodings,
    train_df["label"]
)

val_dataset = DocumentDataset(
    val_encodings,
    val_df["label"]
)

test_dataset = DocumentDataset(
    test_encodings,
    test_df["label"]
)

print("Datasets created successfully!")
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

Datasets created successfully!
Training samples: 1728
Validation samples: 370
Testing samples: 371


In [8]:
NUM_LABELS = 3

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)

print("Base model loaded!")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model loaded!


In [9]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none"
)

print("LoRA configuration created!")

LoRA configuration created!


In [10]:
model = get_peft_model(model, lora_config)

print("LoRA applied successfully!")

LoRA applied successfully!


In [11]:
model.print_trainable_parameters()

trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

print("Metrics function ready!")

Metrics function ready!


In [13]:
training_args = TrainingArguments(
    output_dir="../results/lora_checkpoints",

    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="no",

    logging_steps=100,

    report_to="none",

    use_cpu=True
)

print("Training configuration ready!")

Training configuration ready!


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics
)

print("Trainer ready!")

Trainer ready!


In [15]:
train_result = trainer.train()

print("LoRA training complete!")

  0%|          | 0/648 [00:00<?, ?it/s]

{'loss': 1.0047, 'grad_norm': 2.879925489425659, 'learning_rate': 1.6913580246913582e-05, 'epoch': 0.46}
{'loss': 0.671, 'grad_norm': 1.955716609954834, 'learning_rate': 1.3827160493827162e-05, 'epoch': 0.93}


  0%|          | 0/47 [00:00<?, ?it/s]

{'eval_loss': 0.3219163417816162, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_runtime': 60.2292, 'eval_samples_per_second': 6.143, 'eval_steps_per_second': 0.78, 'epoch': 1.0}
{'loss': 0.2339, 'grad_norm': 0.5797132849693298, 'learning_rate': 1.0740740740740742e-05, 'epoch': 1.39}
{'loss': 0.057, 'grad_norm': 0.43579337000846863, 'learning_rate': 7.654320987654322e-06, 'epoch': 1.85}


  0%|          | 0/47 [00:00<?, ?it/s]

{'eval_loss': 0.02080933004617691, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_runtime': 60.6969, 'eval_samples_per_second': 6.096, 'eval_steps_per_second': 0.774, 'epoch': 2.0}
{'loss': 0.0261, 'grad_norm': 0.2977953851222992, 'learning_rate': 4.567901234567902e-06, 'epoch': 2.31}
{'loss': 0.0178, 'grad_norm': 0.218121737241745, 'learning_rate': 1.4814814814814815e-06, 'epoch': 2.78}


  0%|          | 0/47 [00:00<?, ?it/s]

{'eval_loss': 0.011765304952859879, 'eval_accuracy': 1.0, 'eval_macro_f1': 1.0, 'eval_runtime': 60.0771, 'eval_samples_per_second': 6.159, 'eval_steps_per_second': 0.782, 'epoch': 3.0}
{'train_runtime': 3778.8788, 'train_samples_per_second': 1.372, 'train_steps_per_second': 0.171, 'train_loss': 0.3115135955589789, 'epoch': 3.0}
LoRA training complete!


In [16]:
test_results = trainer.evaluate(test_dataset)

print("LoRA Test Results:")

for key, value in test_results.items():
    if key.startswith("eval_"):
        print(f"{key}: {value}")

  0%|          | 0/47 [00:00<?, ?it/s]

LoRA Test Results:
eval_loss: 0.011540445499122143
eval_accuracy: 1.0
eval_macro_f1: 1.0
eval_runtime: 62.0047
eval_samples_per_second: 5.983
eval_steps_per_second: 0.758


In [18]:
label_names = [
    "Invoices",
    "Purchase Orders",
    "Shipping Orders"
]

In [19]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

predictions = trainer.predict(test_dataset)

predicted_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=label_names
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        true_labels,
        predicted_labels
    )
)

  0%|          | 0/47 [00:00<?, ?it/s]

                 precision    recall  f1-score   support

       Invoices       1.00      1.00      1.00       125
Purchase Orders       1.00      1.00      1.00       125
Shipping Orders       1.00      1.00      1.00       121

       accuracy                           1.00       371
      macro avg       1.00      1.00      1.00       371
   weighted avg       1.00      1.00      1.00       371

Confusion Matrix:
[[125   0   0]
 [  0 125   0]
 [  0   0 121]]


In [20]:
import pandas as pd

results = pd.DataFrame({
    "Method": [
        "Full Fine-Tuning",
        "LoRA"
    ],
    "Test Accuracy": [
        1.0,
        1.0
    ],
    "Macro F1": [
        1.0,
        1.0
    ],
    "Trainable Parameters": [
        66955779,
        740355
    ],
    "Trainable Percentage": [
        100.0,
        1.0936
    ],
    "Training Time Seconds": [
        4676.2522,
        3778.8788
    ]
})

results

,Method,Test Accuracy,Macro F1,Trainable Parameters,Trainable Percentage,Training Time Seconds
0,Full Fine-Tuning,1.0,1.0,66955779,100.0000,4676.2522
1,LoRA,1.0,1.0,740355,1.0936,3778.8788


In [21]:
results.to_csv(
    "../results/baseline_vs_lora.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!
